# Session 2 · Creating Textual Content with Generative AI

Generative AI › Track 3 · Innovate with Generative AI

Prepared by **Hamna Zahid Khan**, Junior Lecturer

Run each cell from top to bottom (Runtime ▸ Run all in Colab). Outputs below were produced by actually executing this notebook.

### Example · `session-02_prompt_builder.py`

In [1]:
"""
Session 2 · Example 1 — A reusable prompt builder for written content.

A good writing prompt has six parts: Role, Task, Audience, Context,
Format and Constraints (the "RTACFC" checklist used in this session).
This class assembles them and refuses to build a prompt with gaps,
the same way a careful writer checks a brief before starting.

Run:  python session-02_prompt_builder.py
"""
from dataclasses import dataclass, field


@dataclass
class WritingPrompt:
    role: str = ""
    task: str = ""
    audience: str = ""
    context: str = ""
    fmt: str = ""
    constraints: list = field(default_factory=list)

    REQUIRED = ("role", "task", "audience", "fmt")

    def missing(self) -> list:
        """Names of the required parts that are still empty."""
        return [name for name in self.REQUIRED if not getattr(self, name).strip()]

    def build(self) -> str:
        gaps = self.missing()
        if gaps:
            raise ValueError(f"Prompt is incomplete, please fill: {', '.join(gaps)}")
        lines = [
            f"You are {self.role}.",
            f"Task: {self.task}",
            f"Audience: {self.audience}",
        ]
        if self.context:
            lines.append(f'Context: """{self.context}"""')
        lines.append(f"Format: {self.fmt}")
        if self.constraints:
            lines.append("Rules:")
            lines += [f"- {c}" for c in self.constraints]
        return "\n".join(lines)


# 1) A vague prompt is caught before it wastes a generation.
vague = WritingPrompt(task="write a blog about AI")
try:
    vague.build()
except ValueError as err:
    print("Caught:", err)

# 2) A complete prompt for a real assignment.
blog = WritingPrompt(
    role="a friendly tech writer who explains ideas with everyday examples",
    task="Write a 600-word blog post titled 'Five ways AI helps first-year students study smarter'.",
    audience="first-year university students in Karachi, new to AI tools",
    context="The campus allows AI for brainstorming and feedback, not for writing graded work.",
    fmt="H1 title, 80-word intro, five H2 sections with one example each, 60-word conclusion with a call to action",
    constraints=["Plain English, sentences under 20 words", "Mention academic integrity once",
                 "No made-up statistics; write [SOURCE NEEDED] if a number is required"],
)
print("\n" + blog.build())

Caught: Prompt is incomplete, please fill: role, audience, fmt

You are a friendly tech writer who explains ideas with everyday examples.
Task: Write a 600-word blog post titled 'Five ways AI helps first-year students study smarter'.
Audience: first-year university students in Karachi, new to AI tools
Context: """The campus allows AI for brainstorming and feedback, not for writing graded work."""
Format: H1 title, 80-word intro, five H2 sections with one example each, 60-word conclusion with a call to action
Rules:
- Plain English, sentences under 20 words
- Mention academic integrity once
- No made-up statistics; write [SOURCE NEEDED] if a number is required


### Example · `session-02_readability.py`

In [2]:
"""
Session 2 · Example 2 — Measure readability of an AI draft vs an edited draft.

The Flesch Reading Ease score (0-100, higher = easier) is a classic formula:
    206.835 - 1.015 * (words / sentences) - 84.6 * (syllables / words)
60-70 is "plain English"; under 30 reads like an academic paper.
AI drafts often come out long-winded, so measuring before and after editing
shows whether your "make it simpler" prompt actually worked.

Run:  python session-02_readability.py
"""
import re


def count_syllables(word: str) -> int:
    """Approximate syllables: count vowel groups, drop a silent final 'e'."""
    word = word.lower()
    groups = re.findall(r"[aeiouy]+", word)
    n = len(groups)
    if word.endswith("e") and n > 1 and not word.endswith("le"):
        n -= 1
    return max(n, 1)


def readability(text: str) -> dict:
    sentences = max(len(re.findall(r"[.!?]+", text)), 1)
    words = re.findall(r"[A-Za-z']+", text)
    syllables = sum(count_syllables(w) for w in words)
    wps = len(words) / sentences
    spw = syllables / len(words)
    score = 206.835 - 1.015 * wps - 84.6 * spw
    return {"words": len(words), "sentences": sentences,
            "words_per_sentence": round(wps, 1), "flesch": round(score, 1)}


ai_draft = ("Artificial intelligence technologies are increasingly being utilised by "
            "contemporary university students in order to facilitate the comprehension of "
            "complicated academic material, consequently improving educational outcomes "
            "significantly across numerous disciplines.")

edited = ("Many students now use AI to understand hard topics. "
          "It can explain an idea in simple words. "
          "It can quiz you before a test. "
          "Used well, it helps you learn faster.")

for label, text in (("AI first draft", ai_draft), ("Edited version", edited)):
    r = readability(text)
    print(f"{label:15} words={r['words']:3} sentences={r['sentences']} "
          f"words/sentence={r['words_per_sentence']:5} Flesch={r['flesch']}")

print("\nGuide: 90+ very easy | 60-70 plain English | 30-50 difficult | <30 very difficult")
print("Very dense text can even score below 0: one 29-word sentence full of long words does that.")

AI first draft  words= 29 sentences=1 words/sentence= 29.0 Flesch=-82.2
Edited version  words= 31 sentences=4 words/sentence=  7.8 Flesch=84.3

Guide: 90+ very easy | 60-70 plain English | 30-50 difficult | <30 very difficult
Very dense text can even score below 0: one 29-word sentence full of long words does that.


### Example · `session-02_platform_fit.py`

In [3]:
"""
Session 2 · Example 3 — Check social posts against platform rules before posting.

AI happily writes a 400-character post for a platform that cuts off at 280,
or stuffs in 15 hashtags. This checker flags such problems automatically.
Limits below are the commonly published ones at the time of writing; check
each platform's help page because they change.

Run:  python session-02_platform_fit.py
"""
import re

RULES = {
    # platform: (max characters, recommended max hashtags)
    "X (Twitter)": (280, 2),
    "Instagram":   (2200, 5),
    "LinkedIn":    (3000, 3),
    "Facebook":    (63206, 3),
}


def check(platform: str, post: str) -> list:
    limit, max_tags = RULES[platform]
    problems = []
    if len(post) > limit:
        problems.append(f"too long: {len(post)}/{limit} characters")
    tags = re.findall(r"#\w+", post)
    if len(tags) > max_tags:
        problems.append(f"{len(tags)} hashtags (keep to {max_tags} or fewer)")
    if not re.search(r"(register|join|read|sign up|link|visit|comment)", post, re.I):
        problems.append("no call to action")
    return problems


posts = {
    "X (Twitter)": "Our AI Study Skills workshop is this Thursday 3 pm in Lab 4! Learn to quiz yourself "
                   "with AI, build flashcards, and stay within the integrity rules. Free for all "
                   "first-years, snacks included, bring your laptop and your questions, see you there "
                   "#AI #StudySmart #Karachi #Students",
    "LinkedIn":    "Proud to host our first AI Study Skills workshop for first-year students. "
                   "We covered tutor-mode prompting, flashcards and academic integrity. "
                   "Read the full recap at the link in comments. #AI #Education",
}

for platform, post in posts.items():
    issues = check(platform, post)
    status = "OK" if not issues else "FIX -> " + "; ".join(issues)
    print(f"{platform:12} {len(post):4} chars  {status}")

X (Twitter)   277 chars  FIX -> 4 hashtags (keep to 2 or fewer); no call to action
LinkedIn      201 chars  OK


### Example · `session-02_project_blog_pipeline.py`

In [4]:
"""
Session 2 · Mini project (full solution) — A blog-writing pipeline.

Real-life problem: you must publish a 700-word, search-friendly blog post
every week for the department website. Writing one giant prompt gives a
generic article. A better workflow splits the job into steps:

  1. Outline  -> a word budget for every section
  2. Section prompts -> one focused prompt per section (better quality)
  3. Quality gate -> check the assembled draft: length, keyword use,
     readability, unfinished placeholders

This script builds the prompts and runs the quality gate on a sample draft.
Run:  python session-02_project_blog_pipeline.py
"""
import re

TITLE = "How to Use AI as a Study Partner (Without Cheating)"
KEYWORD = "ai study partner"
TARGET_WORDS = 700
SECTIONS = [  # (heading, share of the word budget)
    ("Introduction", 0.12),
    ("What an AI study partner can and cannot do", 0.22),
    ("Three prompts that make you think", 0.30),
    ("Staying inside the integrity rules", 0.22),
    ("Conclusion and next step", 0.14),
]

# ---------- Step 1: outline with word budgets ----------
print(f"OUTLINE for '{TITLE}' ({TARGET_WORDS} words)")
budget = {h: round(TARGET_WORDS * share) for h, share in SECTIONS}
for h, w in budget.items():
    print(f"  {w:4} words  {h}")
print(f"  ----\n  {sum(budget.values()):4} words total")

# ---------- Step 2: one prompt per section ----------
print("\nSECTION PROMPTS")
for i, (h, w) in enumerate(budget.items(), 1):
    prompt = (f"[{i}/{len(budget)}] Write the section '{h}' of the blog '{TITLE}'. "
              f"About {w} words, audience: first-year students, friendly tone. "
              f"Use the phrase '{KEYWORD}' at most once. End without a summary line.")
    print("  " + prompt)

# ---------- Step 3: quality gate on an assembled draft ----------
draft = """
# How to Use AI as a Study Partner (Without Cheating)
An AI study partner can quiz you, explain hard ideas, and check your reasoning.
It cannot sit your exam. Studies show students who test themselves remember more [SOURCE NEEDED].
## Three prompts that make you think
Ask: give me one hint, not the answer. Ask: quiz me with five questions. Ask: find the gap in my explanation.
Your AI study partner works best when you do the thinking. Your AI study partner should never write graded work.
"""


def quality_gate(text: str) -> list:
    report = []
    words = re.findall(r"[A-Za-z']+", text)
    report.append(("length", len(words) >= 0.9 * TARGET_WORDS, f"{len(words)}/{TARGET_WORDS} words"))
    hits = text.lower().count(KEYWORD)
    density = 100 * hits * len(KEYWORD.split()) / max(len(words), 1)
    report.append(("keyword density", 0.5 <= density <= 2.5, f"{hits} uses = {density:.1f}% (aim 0.5-2.5%)"))
    placeholders = re.findall(r"\[[A-Z ]+\]", text)
    report.append(("placeholders", not placeholders, f"found {placeholders}" if placeholders else "none"))
    found = [h for h, _ in SECTIONS if h.lower() in text.lower()]
    report.append(("sections", len(found) == len(SECTIONS), f"{len(found)}/{len(SECTIONS)} headings present"))
    return report


print("\nQUALITY GATE")
results = quality_gate(draft)
for name, ok, detail in results:
    print(f"  [{'PASS' if ok else 'FAIL'}] {name:16} {detail}")
passed = sum(ok for _, ok, _ in results)
print(f"\n{passed}/{len(results)} checks passed -> {'publish' if passed == len(results) else 'send back for revision'}")

# ---------- Step 4: turn failures into a precise revision prompt ----------
failed = [f"- {name}: {detail}" for name, ok, detail in results if not ok]
if failed:
    print("\nREVISION PROMPT (paste back into the assistant)")
    print("Revise the draft below. Fix ONLY these problems and keep everything else:")
    print("\n".join(failed))
    print("Replace any [SOURCE NEEDED] with a real, named source or delete the claim.")

OUTLINE for 'How to Use AI as a Study Partner (Without Cheating)' (700 words)
    84 words  Introduction
   154 words  What an AI study partner can and cannot do
   210 words  Three prompts that make you think
   154 words  Staying inside the integrity rules
    98 words  Conclusion and next step
  ----
   700 words total

SECTION PROMPTS
  [1/5] Write the section 'Introduction' of the blog 'How to Use AI as a Study Partner (Without Cheating)'. About 84 words, audience: first-year students, friendly tone. Use the phrase 'ai study partner' at most once. End without a summary line.
  [2/5] Write the section 'What an AI study partner can and cannot do' of the blog 'How to Use AI as a Study Partner (Without Cheating)'. About 154 words, audience: first-year students, friendly tone. Use the phrase 'ai study partner' at most once. End without a summary line.
  [3/5] Write the section 'Three prompts that make you think' of the blog 'How to Use AI as a Study Partner (Without Cheating)'. About 2